In [ ]:
!pip install plotly nbformat

## Run

### loading the model

In [1]:
import cv2
import numpy as np
from notebook.utils import setup_sam_3d_body
from tools.vis_utils import visualize_sample_together

# Set up the estimator
estimator = setup_sam_3d_body(hf_repo_id="facebook/sam-3d-body-dinov3")

/workspace/sam-3d-body-measurement/sam_3d_body/models/heads/mhr_head.py:33: UserWarning: Momentum is not enabled
  warnings.warn("Momentum is not enabled")


Loading SAM 3D Body model from facebook/sam-3d-body-dinov3...


/root/miniconda3/envs/sam_3d_body/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Fetching 5 files: 100%|██████████| 5/5 [00:00<00:00, 31726.96it/s]


Loading SAM 3D Body model...


Using cache found in /root/.cache/torch/hub/facebookresearch_dinov3_main
Ignored kwargs: {'drop_path': 0.1}
The model and loaded state dict do not match exactly

missing keys in source state_dict: backbone.encoder.mask_token, head_pose.hand_pose_comps_ori, head_pose.mhr.face_expressions_model.shape_vectors, head_pose.mhr.pose_correctives_model.pose_dirs_predictor.0.sparse_indices, head_pose.mhr.pose_correctives_model.pose_dirs_predictor.0.sparse_weight, head_pose.mhr.pose_correctives_model.pose_dirs_predictor.2.weight, head_pose.mhr.character_torch.skeleton.joint_translation_offsets, head_pose.mhr.character_torch.skeleton.joint_prerotations, head_pose.mhr.character_torch.skeleton.pmi, head_pose.mhr.character_torch.skeleton.joint_parents, head_pose.mhr.character_torch.mesh.rest_vertices, head_pose.mhr.character_torch.mesh.faces, head_pose.mhr.character_torch.mesh.texcoords, head_pose.mhr.character_torch.mesh.texcoord_faces, head_pose.mhr.character_torch.parameter_transform.parameter_tra

Loading human detector from vitdet...
########### Using human detector: ViTDet...


/root/miniconda3/envs/sam_3d_body/lib/python3.11/site-packages/detectron2/config/lazy.py:167: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  return old_import(name, globals, locals, fromlist=fromlist, level=level)
/root/miniconda3/envs/sam_3d_body/lib/python3.11/site-packages/timm/models/layers/__init__.py:48: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)


Loading FOV estimator from moge2...
########### Using fov estimator: MoGe2...
Mask-condition inference is not supported...
Setup complete!
  Human detector: ✓
  Human segmentor: ✗ (mask inference disabled)
  FOV estimator: ✓


### inference

In [2]:
# Load and process image
img_bgr = cv2.imread("/workspace/sam-3d-body-measurement/notebook/images/r.png")
outputs = estimator.process_one_image(cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB))


####### Please make sure the input image is in RGB format
Running object detector...


/root/miniconda3/envs/sam_3d_body/lib/python3.11/site-packages/torch/functional.py:505: UserWarning: torch.meshgrid: in an upcoming release, it will be required to pass the indexing argument. (Triggered internally at /pytorch/aten/src/ATen/native/TensorShape.cpp:4317.)
  return _VF.meshgrid(tensors, **kwargs)  # type: ignore[attr-defined]
W1122 16:37:45.308000 2450 site-packages/torch/fx/_symbolic_trace.py:52] is_fx_tracing will return true for both fx.symbolic_trace and torch.export. Please use is_fx_tracing_symbolic_tracing() for specifically fx.symbolic_trace or torch.compiler.is_compiling() for specifically torch.export/compile.


Found boxes: [[  74.628975   50.703465  596.51324  1092.9524  ]]
Running FOV estimator ...


### Experiments

#### measure

In [ ]:
import torch
import trimesh
import numpy as np

def get_tpose_mesh_final(estimator, outputs):
    # --- 1. Locate the MHR Layer ---
    # Based on your previous logs:
    mhr_layer = estimator.model.head_pose.mhr
    device = next(mhr_layer.parameters()).device
    
    # --- 2. Prepare Tensors (Corrected Dimensions) ---
    
    # A. Identity (Shape)
    # Taken from inference output. Size: [1, 45]
    raw_shape = outputs[0]['shape_params']
    id_coeffs = torch.tensor(raw_shape).float().to(device).unsqueeze(0)
    
    # B. Pose (Model Parameters) - THE FIX
    # The error proved the model expects 249 total inputs.
    # 249 - 45 (Shape) = 204 (Pose).
    # We must provide 204 zeros for the T-Pose.
    pose_coeffs = torch.zeros((1, 204)).float().to(device)
    
    # C. Face Expression
    # Standard MHR size: 72
    face_coeffs = torch.zeros((1, 72)).float().to(device)

    # --- 3. Generate Mesh ---
    # We print shapes to confirm they sum to 249 (204+45)
    print(f"Generating T-Pose with: Shape={id_coeffs.shape[1]} + Pose={pose_coeffs.shape[1]} = {id_coeffs.shape[1]+pose_coeffs.shape[1]}")
    
    with torch.no_grad():
        # Positional arguments: identity, pose, face, correctives
        res = mhr_layer(
            id_coeffs,      
            pose_coeffs,    
            face_coeffs,    
            True            
        )

    # --- 4. Parse Output ---
    # Unpack the tuple ((verts, joints)) structure
    if isinstance(res, tuple):
        if isinstance(res[0], (tuple, list)):
            verts = res[0][0]
            joints = res[0][1]
        else:
            verts = res[0]
            joints = res[1]
    else:
        # Fallback for single return
        verts = res
        joints = None

    # --- 5. Get Faces ---
    # Fallback to standard MHR face logic if attribute is hidden
    try:
        if hasattr(mhr_layer, 'character_torch'):
            faces = mhr_layer.character_torch.mesh.faces
        else:
            # Try the path from your previous logs
            faces = estimator.model.head_pose.mhr.character_torch.mesh.faces
        faces_np = faces.cpu().numpy()
    except:
        print("Warning: Could not find faces array. Exporting Point Cloud only.")
        faces_np = None

    # --- 6. Return Trimesh ---
    verts_np = verts.cpu().numpy()[0]
    
    # Create the mesh
    mesh = trimesh.Trimesh(vertices=verts_np, faces=faces_np)
    
    # Return joints if available, otherwise we might need to recalculate them
    if joints is not None:
        joints_np = joints.cpu().numpy()[0]
    else:
        joints_np = None
        
    return mesh, joints_np

# # --- Execution ---
try:
    # 1. Run the generator
    tpose_mesh, tpose_joints = get_tpose_mesh_final(estimator, outputs)
    print(f"\nSUCCESS: Generated T-Pose Mesh.")
    print(f"Vertices: {len(tpose_mesh.vertices)}")
    print(f"Joints: {len(tpose_joints) if tpose_joints is not None else 0}")

except Exception as e:
    print(f"Error: {e}")

Generating T-Pose with: Shape=45 + Pose=204 = 249

SUCCESS: Generated T-Pose Mesh.
Vertices: 18439
Joints: 127


#### visualize

In [40]:
import plotly.graph_objects as go
import numpy as np
import trimesh

# Your specific joint mapping
idx_map = {
    "pelvis": 1,
    "waist": 35,
    "spine_upper": 37,
    "spine_lower": 36,
    "neck": 110,
    "r_shoulder": 69,
    "r_wrist": 60
}

def analyze_and_visualize_3d(mesh, joints, target_height_cm=163.0):
    print(f"--- STARTING ANALYSIS (Target Height: {target_height_cm}cm) ---")

    # ==========================================
    # 1. SCALE CORRECTION
    # ==========================================
    # Calculate current height (Head to Toe using vertices)
    current_height = mesh.vertices[:, 1].max() - mesh.vertices[:, 1].min()
    scale_factor = target_height_cm / current_height

    print(f"Resizing Mesh... (Scale Factor: {scale_factor:.4f})")
    mesh.vertices *= scale_factor
    joints *= scale_factor

    # ==========================================
    # 2. MEASUREMENT LOGIC
    # ==========================================
    # Define Heights
    y_chest = (joints[idx_map["spine_upper"]][1] + joints[idx_map["spine_lower"]][1]) / 2.0 + 3
    y_waist = joints[idx_map["waist"]][1]
    y_hips  = joints[idx_map["pelvis"]][1]

    # Helper function: Gets circumference of LARGEST loop only (ignores arms)
    def get_clean_circ(height_y):
        slice_obj = mesh.section(plane_origin=[0, height_y, 0], plane_normal=[0, 1, 0])
        if not slice_obj: return 0.0, None

        max_len = 0.0
        # Iterate over all loops (arms, torso, noise)
        for loop_points in slice_obj.discrete:
            # Calculate perimeter
            loop_len = np.sum(np.sqrt(np.sum(np.diff(loop_points, axis=0)**2, axis=1)))
            loop_len += np.linalg.norm(loop_points[-1] - loop_points[0]) # Close loop
            
            if loop_len > max_len:
                max_len = loop_len
        return max_len, slice_obj

    # Calculate Circumferences
    chest_circ, chest_slice = get_clean_circ(y_chest)
    waist_circ, waist_slice = get_clean_circ(y_waist)
    hip_circ, hip_slice     = get_clean_circ(y_hips)

    # Calculate Linear Lengths
    p_neck = joints[idx_map["neck"]]
    p_pelvis = joints[idx_map["pelvis"]]
    torso_len = np.linalg.norm(p_neck - p_pelvis)

    p_shoulder = joints[idx_map["r_shoulder"]]
    p_wrist = joints[idx_map["r_wrist"]]
    arm_len = np.linalg.norm(p_shoulder - p_wrist)

    # Print to Console
    print("\n--- MEASUREMENTS ---")
    print(f"Chest: {chest_circ:.2f} cm")
    print(f"Waist: {waist_circ:.2f} cm")
    print(f"Hips:  {hip_circ:.2f} cm")
    print(f"Torso: {torso_len:.2f} cm")
    print(f"Arm:   {arm_len:.2f} cm")

    # ==========================================
    # 3. INTERACTIVE 3D VISUALIZATION
    # ==========================================
    print("\nGenerating 3D Scene...")
    traces = []

    # A. BODY MESH
    x, y, z = mesh.vertices.T
    i, j, k = mesh.faces.T
    traces.append(go.Mesh3d(
        x=x, y=y, z=z, i=i, j=j, k=k,
        color='lightgray', opacity=0.3, name='Body Skin', hoverinfo='skip'
    ))

    # B. SLICES (Chest, Waist, Hips)
    def add_slice_trace(slice_obj, color, name):
        if slice_obj:
            for path in slice_obj.discrete:
                traces.append(go.Scatter3d(
                    x=path[:, 0], y=path[:, 1], z=path[:, 2],
                    mode='lines', line=dict(color=color, width=5),
                    name=name, showlegend=False
                ))
    
    add_slice_trace(chest_slice, 'blue', 'Chest Slice')
    add_slice_trace(waist_slice, 'green', 'Waist Slice')
    add_slice_trace(hip_slice, 'purple', 'Hip Slice')

    # C. SKELETON LINES & LABELS
    
    # 1. Spine Line (Orange)
    spine_indices = [idx_map['neck'], idx_map['spine_upper'], idx_map['spine_lower'], idx_map['waist'], idx_map['pelvis']]
    spine_pts = joints[spine_indices]
    traces.append(go.Scatter3d(
        x=spine_pts[:, 0], y=spine_pts[:, 1], z=spine_pts[:, 2],
        mode='lines+markers', line=dict(color='orange', width=5), marker=dict(size=4),
        name='Spine Length'
    ))
    
    # Spine Label
    mid_torso = (p_neck + p_pelvis) / 2
    traces.append(go.Scatter3d(
        x=[mid_torso[0] + 10], y=[mid_torso[1]], z=[mid_torso[2]], # Offset slightly X
        mode='text', text=[f"Torso: {torso_len:.1f}cm"],
        textfont=dict(color='orange', size=12, family="Arial Black"),
        name='Torso Label', showlegend=False
    ))

    # 2. Arm Line (Cyan)
    arm_pts = np.array([p_shoulder, p_wrist])
    traces.append(go.Scatter3d(
        x=arm_pts[:, 0], y=arm_pts[:, 1], z=arm_pts[:, 2],
        mode='lines+markers', line=dict(color='cyan', width=5), marker=dict(size=4),
        name='Arm Length'
    ))

    # Arm Label
    mid_arm = (p_shoulder + p_wrist) / 2
    traces.append(go.Scatter3d(
        x=[mid_arm[0]], y=[mid_arm[1] + 5], z=[mid_arm[2]], # Offset slightly Y
        mode='text', text=[f"Arm: {arm_len:.1f}cm"],
        textfont=dict(color='cyan', size=12, family="Arial Black"),
        name='Arm Label', showlegend=False
    ))

    # D. KEY JOINTS (Red Dots)
    relevant_indices = list(idx_map.values())
    rel_joints = joints[relevant_indices]
    traces.append(go.Scatter3d(
        x=rel_joints[:, 0], y=rel_joints[:, 1], z=rel_joints[:, 2],
        mode='markers', marker=dict(size=5, color='red'),
        name='Joints', hovertext=list(idx_map.keys())
    ))

    # Dummy traces for Legend
    traces.append(go.Scatter3d(x=[None], y=[None], z=[None], mode='lines', line=dict(color='blue', width=4), name=f'Chest: {chest_circ:.1f}cm'))
    traces.append(go.Scatter3d(x=[None], y=[None], z=[None], mode='lines', line=dict(color='green', width=4), name=f'Waist: {waist_circ:.1f}cm'))

    # ==========================================
    # 4. PLOT SETUP
    # ==========================================
    fig = go.Figure(data=traces)
    fig.update_layout(
        title=f"3D Body Analysis (Height: {target_height_cm}cm)",
        scene=dict(
            xaxis=dict(visible=False),
            yaxis=dict(visible=False),
            zaxis=dict(visible=False),
            aspectmode='data'
        ),
        width=900, height=800,
        margin=dict(r=0, l=0, b=0, t=40),
        legend=dict(yanchor="top", y=0.9, xanchor="left", x=0.1)
    )
    fig.show()

# --- EXECUTE ---
# Ensure tpose_mesh and tpose_joints exist from previous steps
analyze_and_visualize_3d(tpose_mesh, tpose_joints, target_height_cm=163.0)

--- STARTING ANALYSIS (Target Height: 163.0cm) ---
Resizing Mesh... (Scale Factor: 1.0000)

--- MEASUREMENTS ---
Chest: 94.76 cm
Waist: 91.37 cm
Hips:  104.26 cm
Torso: 48.65 cm
Arm:   47.59 cm

Generating 3D Scene...


In [45]:
import plotly.graph_objects as go
import numpy as np
import trimesh
import networkx as nx
import scipy.spatial

# Your specific joint mapping
idx_map = {
    "pelvis": 1,
    "waist": 35,
    "spine_upper": 37,
    "spine_lower": 36,
    "neck": 110,
    "r_shoulder": 69,
    "r_wrist": 60
}

def analyze_and_visualize_3d_surface(mesh, joints_input, target_height_cm=163.0):
    print(f"--- STARTING SURFACE ANALYSIS (Target Height: {target_height_cm}cm) ---")

    # ==========================================
    # 0. SAFETY FIX: ENSURE XYZ ONLY
    # ==========================================
    joints = joints_input.copy()
    if joints.shape[1] > 3:
        joints = joints[:, :3]

    # ==========================================
    # 1. SCALE CORRECTION
    # ==========================================
    current_height = mesh.vertices[:, 1].max() - mesh.vertices[:, 1].min()
    scale_factor = target_height_cm / current_height

    print(f"Resizing Mesh... (Scale Factor: {scale_factor:.4f})")
    mesh.vertices *= scale_factor
    joints *= scale_factor

    # ==========================================
    # 2. PREPARE GRAPH
    # ==========================================
    print("Building adjacency graph for surface measurement...")
    graph = nx.Graph()
    edges = mesh.edges_unique
    
    # Calculate weights (Euclidean distance)
    weights = np.linalg.norm(mesh.vertices[edges[:, 0]] - mesh.vertices[edges[:, 1]], axis=1)
    
    # Add edges individually to ensure Node IDs remain INTEGERS
    for (u, v), w in zip(edges, weights):
        graph.add_edge(u, v, weight=w)
    
    # KDTree for snapping
    tree = scipy.spatial.KDTree(mesh.vertices)

    # Helper: Calculate Geodesic Path
    def get_surface_path(start_point, end_point):
        # 1. Snap input points (XYZ) to nearest mesh vertices
        _, start_idx = tree.query(start_point)
        _, end_idx = tree.query(end_point)
        
        # 2. Find shortest path on the graph
        try:
            path_indices = nx.shortest_path(graph, source=start_idx, target=end_idx, weight='weight')
            
            # 3. Get coordinates
            path_points = mesh.vertices[path_indices]
            
            # 4. Calculate length
            diffs = np.diff(path_points, axis=0)
            length = np.sum(np.sqrt(np.sum(diffs**2, axis=1)))
            return length, path_points
            
        except nx.NetworkXNoPath:
            print(f"Warning: No path found between vertices {start_idx} and {end_idx}")
            return 0.0, None
        except Exception as e:
            print(f"Graph Error: {e}")
            return 0.0, None

    # ==========================================
    # 3. MEASUREMENT LOGIC
    # ==========================================
    # Define Heights
    y_chest = (joints[idx_map["spine_upper"]][1] + joints[idx_map["spine_lower"]][1]) / 2.0 + 3
    y_waist = (joints[idx_map["waist"]][1] + joints[idx_map["spine_lower"]][1]) / 2.0
    y_hips  = joints[idx_map["pelvis"]][1]

    # Circumference Helper
    def get_clean_circ(height_y):
        slice_obj = mesh.section(plane_origin=[0, height_y, 0], plane_normal=[0, 1, 0])
        if not slice_obj: return 0.0, None

        max_len = 0.0
        # Iterate over all loops to find the largest (Torso)
        for loop_points in slice_obj.discrete:
            loop_len = np.sum(np.sqrt(np.sum(np.diff(loop_points, axis=0)**2, axis=1)))
            loop_len += np.linalg.norm(loop_points[-1] - loop_points[0]) 
            if loop_len > max_len:
                max_len = loop_len
        return max_len, slice_obj

    # Calculate Circumferences
    chest_circ, chest_slice = get_clean_circ(y_chest)
    waist_circ, waist_slice = get_clean_circ(y_waist)
    hip_circ, hip_slice     = get_clean_circ(y_hips)

    # --- CALCULATE PATHS ---
    print("Calculating geodesic paths...")
    
    # 1. ARM: Shoulder -> Wrist
    p_shoulder = joints[idx_map["r_shoulder"]]
    p_wrist = joints[idx_map["r_wrist"]]
    arm_len, arm_path = get_surface_path(p_shoulder, p_wrist)

    # 2. TORSO: Neck -> Point on Hip Slice (NEW LOGIC)
    p_neck = joints[idx_map["neck"]]
    
    # We need to find the point on the 'hip_slice' that is closest to the neck
    # This ensures the tape goes down the front/center and stops exactly at the purple line.
    best_hip_point = None
    min_dist = float('inf')

    if hip_slice:
        # hip_slice.discrete is a list of loops (arrays of points)
        for loop in hip_slice.discrete:
            for point in loop:
                # Find point on ring closest to neck (straight line dist)
                # This naturally selects the point on the upper-body side
                dist = np.linalg.norm(point - p_neck)
                if dist < min_dist:
                    min_dist = dist
                    best_hip_point = point
    
    # Fallback if slice failed
    if best_hip_point is None:
        best_hip_point = joints[idx_map["pelvis"]] # Fallback to joint

    torso_len, torso_path = get_surface_path(p_neck, best_hip_point)

    print("\n--- MEASUREMENTS (Surface/Geodesic) ---")
    print(f"Chest: {chest_circ:.2f} cm")
    print(f"Waist: {waist_circ:.2f} cm")
    print(f"Hips:  {hip_circ:.2f} cm")
    print(f"Torso (Surface): {torso_len:.2f} cm")
    print(f"Arm (Surface):   {arm_len:.2f} cm")

    # ==========================================
    # 4. INTERACTIVE 3D VISUALIZATION
    # ==========================================
    print("\nGenerating 3D Scene...")
    traces = []

    # A. BODY MESH
    x, y, z = mesh.vertices.T
    i, j, k = mesh.faces.T
    traces.append(go.Mesh3d(
        x=x, y=y, z=z, i=i, j=j, k=k,
        color='lightgray', opacity=0.3, name='Body Skin', hoverinfo='skip'
    ))

    # B. SLICES
    def add_slice_trace(slice_obj, color, name):
        if slice_obj:
            for path in slice_obj.discrete:
                traces.append(go.Scatter3d(
                    x=path[:, 0], y=path[:, 1], z=path[:, 2],
                    mode='lines', line=dict(color=color, width=5),
                    name=name, showlegend=False
                ))
    
    add_slice_trace(chest_slice, 'blue', 'Chest Slice')
    add_slice_trace(waist_slice, 'green', 'Waist Slice')
    add_slice_trace(hip_slice, 'purple', 'Hip Slice')

    # C. SURFACE PATHS
    if torso_path is not None:
        traces.append(go.Scatter3d(
            x=torso_path[:, 0], y=torso_path[:, 1], z=torso_path[:, 2],
            mode='lines', line=dict(color='orange', width=5), name='Torso Tape'
        ))
        mid = len(torso_path) // 2
        traces.append(go.Scatter3d(
            x=[torso_path[mid, 0] + 5], y=[torso_path[mid, 1]], z=[torso_path[mid, 2] + 10],
            mode='text', text=[f"Torso: {torso_len:.1f}cm"],
            textfont=dict(color='orange', size=12, family="Arial Black"), showlegend=False
        ))

    if arm_path is not None:
        traces.append(go.Scatter3d(
            x=arm_path[:, 0], y=arm_path[:, 1], z=arm_path[:, 2],
            mode='lines', line=dict(color='cyan', width=5), name='Arm Tape'
        ))
        mid = len(arm_path) // 2
        traces.append(go.Scatter3d(
            x=[arm_path[mid, 0]], y=[arm_path[mid, 1] + 5], z=[arm_path[mid, 2] + 5],
            mode='text', text=[f"Arm: {arm_len:.1f}cm"],
            textfont=dict(color='cyan', size=12, family="Arial Black"), showlegend=False
        ))

    # D. JOINTS
    relevant_indices = list(idx_map.values())
    rel_joints = joints[relevant_indices]
    traces.append(go.Scatter3d(
        x=rel_joints[:, 0], y=rel_joints[:, 1], z=rel_joints[:, 2],
        mode='markers', marker=dict(size=5, color='red'),
        name='Joints', hovertext=list(idx_map.keys())
    ))

    # Legend Helpers
    traces.append(go.Scatter3d(x=[None], y=[None], z=[None], mode='lines', line=dict(color='blue', width=4), name=f'Chest: {chest_circ:.1f}cm'))
    traces.append(go.Scatter3d(x=[None], y=[None], z=[None], mode='lines', line=dict(color='green', width=4), name=f'Waist: {waist_circ:.1f}cm'))

    fig = go.Figure(data=traces)
    fig.update_layout(
        title=f"3D Body Surface Analysis (Height: {target_height_cm}cm)",
        scene=dict(xaxis=dict(visible=False), yaxis=dict(visible=False), zaxis=dict(visible=False), aspectmode='data'),
        width=900, height=800, margin=dict(r=0, l=0, b=0, t=40)
    )
    fig.show()

# --- EXECUTE ---
analyze_and_visualize_3d_surface(tpose_mesh, tpose_joints, target_height_cm=163.0)

--- STARTING SURFACE ANALYSIS (Target Height: 163.0cm) ---
Resizing Mesh... (Scale Factor: 1.0000)
Building adjacency graph for surface measurement...
Calculating geodesic paths...

--- MEASUREMENTS (Surface/Geodesic) ---
Chest: 94.76 cm
Waist: 84.28 cm
Hips:  104.26 cm
Torso (Surface): 52.21 cm
Arm (Surface):   50.98 cm

Generating 3D Scene...


### peek joints

In [36]:
import plotly.graph_objects as go
import numpy as np

def visualize_joints_with_indices(joints):
    print(f"Visualizing {len(joints)} joints with indices...")

    # Create list of index strings ("0", "1", "2"...)
    indices = [str(i) for i in range(len(joints))]

    # Create the 3D Scatter plot
    fig = go.Figure(data=[go.Scatter3d(
        x=joints[:, 0],
        y=joints[:, 1],
        z=joints[:, 2],
        mode='markers+text',       # Show both Dots and Numbers
        text=indices,              # The labels are the indices
        textposition="top center", # Put number slightly above the dot

        # Styling the Text
        textfont=dict(
            size=9,                # Keep small to reduce clutter
            color='black'
        ),

        # Styling the Dots
        marker=dict(
            size=5,
            color=np.arange(len(joints)), # Color gradient helps distinguish order
            colorscale='Jet',
            opacity=0.8
        ),

        # Hover Tooltip
        hoverinfo='text',
        hovertext=[f"Joint Index: {i}" for i in range(len(joints))]
    )])

    # Layout formatting
    fig.update_layout(
        title="Joint Index Finder (Scroll to Zoom, Drag to Rotate)",
        scene=dict(
            xaxis=dict(visible=False),
            yaxis=dict(visible=False),
            zaxis=dict(visible=False),
            aspectmode='data' # Keeps human proportions correct
        ),
        width=1000,
        height=800,
        margin=dict(r=0, l=0, b=0, t=40)
    )

    fig.show()

# --- RUN IT ---
# Use the joints from your generated T-Pose
if 'tpose_joints' in locals() and tpose_joints is not None:
    visualize_joints_with_indices(tpose_joints)
else:
    print("Error: 'tpose_joints' variable not found. Please run the T-Pose generator first.")

Visualizing 127 joints with indices...


## Pipeline

In [35]:
def pipe(model, file, height):
    img_bgr = cv2.imread(file)
    outputs = model.process_one_image(cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB))

    try:
        # 1. Run the generator
        tpose_mesh, tpose_joints = get_tpose_mesh_final(estimator, outputs)
        print(f"\nSUCCESS: Generated T-Pose Mesh.")
        print(f"Vertices: {len(tpose_mesh.vertices)}")
        print(f"Joints: {len(tpose_joints) if tpose_joints is not None else 0}")

    except Exception as e:
        print(f"Error: {e}")
    
    analyze_and_visualize_3d(tpose_mesh, tpose_joints, target_height_cm=height)

# pipe(estimator, "/workspace/sam-3d-body-measurement/notebook/images/r.png", 163.0)